# Knowable — Fine-Tuning Data & Model Outputs

This notebook explores the dataset used to distill Claude Sonnet into Gemma 4 E4B for the [Knowable](https://github.com/samitizerxu/knowable) macOS tutoring app, and compares model outputs between the base and fine-tuned models.

**Pipeline overview:**
1. The Knowable app captures camera frames of handwritten math + Sonnet's Socratic responses
2. These traces are stored in S3 and assembled into a training dataset
3. A LoRA adapter is trained on Gemma 4 E4B to reproduce Sonnet's output format
4. The fine-tuned model runs locally via Ollama for on-device inference

## 1. Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -r notebook_requirements.txt

In [ ]:
import json
import textwrap
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, HTML, Markdown
from PIL import Image

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"

## 2. Download the dataset

Traces are captured to S3 and assembled into `dataset.jsonl` + `frames/` by `build_dataset.py`.

**Option A — Sync from public S3 and build locally** (no credentials needed):

In [ ]:
# Option A: sync from public S3 and build the dataset (no AWS credentials needed)
# !aws s3 sync s3://knowable-finetune-traces-public ./raw --no-sign-request
# !python build_dataset.py --raw ./raw --out ./dataset

**Option B — Use an existing local dataset** (already built on Lambda or copied locally):

In [ ]:
DATASET_ROOT = Path("./dataset")

assert (DATASET_ROOT / "dataset.jsonl").exists(), (
    f"dataset.jsonl not found at {DATASET_ROOT}. "
    "Either sync from S3 (Option A above) or set DATASET_ROOT to your local copy."
)

## 3. Load & explore the training data

In [ ]:
rows = []
with open(DATASET_ROOT / "dataset.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))

print(f"Total traces: {len(rows)}")

### 3.1 Dataset statistics

In [ ]:
def extract_stats(row):
    user_msg = next(m for m in row["messages"] if m["role"] == "user")
    asst_msg = next(m for m in row["messages"] if m["role"] == "assistant")
    sys_msg = next(m for m in row["messages"] if m["role"] == "system")

    images = [p for p in user_msg["content"] if p["type"] == "image"]
    user_text = next((p["text"] for p in user_msg["content"] if p["type"] == "text"), "")
    asst_text = next((p["text"] for p in asst_msg["content"] if p["type"] == "text"), "")
    sys_text = next((p["text"] for p in sys_msg["content"] if p["type"] == "text"), "")

    return {
        "trace_id": row.get("trace_id", "unknown"),
        "num_images": len(images),
        "user_text_len": len(user_text),
        "assistant_text_len": len(asst_text),
        "system_text_len": len(sys_text),
        "has_hint": "HINT:" in asst_text or "HINT_SPEECH:" in asst_text,
    }

stats = pd.DataFrame([extract_stats(r) for r in rows])
stats.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

stats["num_images"].value_counts().sort_index().plot.bar(ax=axes[0], color="#4A90D9")
axes[0].set_title("Frames per trace")
axes[0].set_xlabel("Number of images")
axes[0].set_ylabel("Count")

stats["assistant_text_len"].hist(ax=axes[1], bins=20, color="#50C878")
axes[1].set_title("Response length (chars)")
axes[1].set_xlabel("Characters")

stats["has_hint"].value_counts().plot.bar(ax=axes[2], color=["#E8E8E8", "#FF6B6B"])
axes[2].set_title("Traces with hints")
axes[2].set_xticklabels(["No hint", "Has hint"], rotation=0)

plt.tight_layout()
plt.show()

### 3.2 Browse individual traces

Each trace has:
- **System prompt** — the Milo persona instructions (shared across all traces)
- **User turn** — camera frame(s) + flags (is_milo_speaking, force_reply, user_query) + current_analysis + event_log
- **Assistant turn** — Sonnet's response with structured sections: UNDERSTANDING, EVENTS, HINT, HINT_SPEECH, STATE

In [ ]:
def show_trace(row, dataset_root, show_system=False):
    """Render a single trace: images, user text, and Sonnet's response."""
    trace_id = row.get("trace_id", "unknown")
    display(Markdown(f"---\n### Trace `{trace_id}`"))

    for msg in row["messages"]:
        role = msg["role"]

        if role == "system":
            if show_system:
                text = msg["content"][0]["text"]
                display(Markdown(f"**System prompt** ({len(text)} chars):"))
                print(text[:500] + ("\n..." if len(text) > 500 else ""))
            continue

        if role == "user":
            images = [p for p in msg["content"] if p["type"] == "image"]
            text_parts = [p for p in msg["content"] if p["type"] == "text"]

            if images:
                n = len(images)
                fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
                if n == 1:
                    axes = [axes]
                for ax, img_part in zip(axes, images):
                    img = Image.open(dataset_root / img_part["image"])
                    ax.imshow(img)
                    ax.set_title(Path(img_part["image"]).name)
                    ax.axis("off")
                plt.tight_layout()
                plt.show()

            if text_parts:
                display(Markdown("**User input (flags + context):**"))
                print(text_parts[0]["text"])

        if role == "assistant":
            text = msg["content"][0]["text"]
            display(Markdown("**Sonnet response (training target):**"))
            print(text)

In [ ]:
# Show first 3 traces (set show_system=True on the first to see the Milo prompt)
for i, row in enumerate(rows[:3]):
    show_trace(row, DATASET_ROOT, show_system=(i == 0))

### 3.3 System prompt

All traces share the same system prompt that defines the "Milo" tutor persona. Here it is in full:

In [ ]:
sys_text = next(
    p["text"]
    for p in rows[0]["messages"][0]["content"]
    if p["type"] == "text"
)
distinct_prompts = len({r.get("system_prompt_sha256", "") for r in rows})
print(f"System prompt length: {len(sys_text)} chars")
print(f"Distinct prompts across dataset: {distinct_prompts}")
print("---")
print(sys_text)

### 3.4 Response structure breakdown

Sonnet outputs structured sections. Let's see how consistently they appear:

In [ ]:
import re

SECTIONS = ["UNDERSTANDING", "EVENTS", "HINT", "HINT_SPEECH", "STATE"]

section_counts = Counter()
state_values = Counter()

for row in rows:
    asst_text = next(
        p["text"]
        for m in row["messages"] if m["role"] == "assistant"
        for p in m["content"] if p["type"] == "text"
    )
    for section in SECTIONS:
        if f"{section}:" in asst_text:
            section_counts[section] += 1
    state_match = re.search(r"STATE:\s*(\S+)", asst_text)
    if state_match:
        state_values[state_match.group(1)] += 1

print("Section presence across all traces:")
for s in SECTIONS:
    pct = section_counts[s] / len(rows) * 100
    print(f"  {s:16s} {section_counts[s]:4d}/{len(rows)}  ({pct:.0f}%)")

print(f"\nSTATE values:")
for state, count in state_values.most_common():
    print(f"  {state:20s} {count:4d}")

## 4. Model outputs

Compare the fine-tuned Gemma 4 E4B (served by Ollama) against the training targets.

### 4.1 Download the fine-tuned model

Pre-quantized unified GGUFs (with vision support baked in) are on HuggingFace:

In [ ]:
HF_REPO = "samitizerxu/knowable-gemma4-e4b-tuned"
GGUF_FILE = "knowable-peft-q4_K_M.gguf"  # or knowable-unsloth-q4_K_M.gguf
MODEL_DIR = Path("./model")
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
from huggingface_hub import hf_hub_download

gguf_path = MODEL_DIR / GGUF_FILE
if not gguf_path.exists():
    print(f"Downloading {GGUF_FILE} from {HF_REPO}...")
    hf_hub_download(
        repo_id=HF_REPO,
        filename=GGUF_FILE,
        local_dir=str(MODEL_DIR),
    )
    print(f"Downloaded to {gguf_path}")
else:
    print(f"Already downloaded: {gguf_path}")

print(f"Size: {gguf_path.stat().st_size / 1e9:.1f} GB")

### 4.2 Create the Ollama model

Register the GGUF with Ollama (Ollama must be running: `ollama serve`):

In [ ]:
OLLAMA_MODEL_NAME = "knowable-tuned"

modelfile = MODEL_DIR / "Modelfile"
modelfile.write_text(
    f"FROM ./{GGUF_FILE}\n"
    f"PARAMETER temperature 0.4\n"
    f"PARAMETER num_ctx 8192\n"
)
print(f"Wrote {modelfile}")
print(f"Run: ollama create {OLLAMA_MODEL_NAME} -f {modelfile}")

In [ ]:
import subprocess

result = subprocess.run(
    ["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(modelfile)],
    cwd=str(MODEL_DIR),
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

### 4.3 Run inference on held-out traces

We send the same inputs (system prompt + images + user text) to the fine-tuned model via Ollama's API and compare against Sonnet's reference response.

In [ ]:
import base64
import requests

OLLAMA_URL = "http://localhost:11434"

def image_to_base64(path: Path) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def run_ollama(row, dataset_root, model_name=OLLAMA_MODEL_NAME):
    """Send a trace's input to Ollama and return the model's response."""
    messages = []

    for msg in row["messages"]:
        if msg["role"] == "assistant":
            continue

        ollama_msg = {"role": msg["role"]}
        images = []
        text = ""

        for part in msg["content"]:
            if part["type"] == "image":
                images.append(image_to_base64(dataset_root / part["image"]))
            elif part["type"] == "text":
                text = part["text"]

        ollama_msg["content"] = text
        if images:
            ollama_msg["images"] = images
        messages.append(ollama_msg)

    resp = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model_name, "messages": messages, "stream": False},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]

In [ ]:
# Use the last N traces as held-out examples
N_EVAL = 5
eval_rows = rows[-N_EVAL:]
print(f"Running inference on {len(eval_rows)} held-out traces...")

In [ ]:
results = []

for i, row in enumerate(eval_rows):
    trace_id = row.get("trace_id", f"row-{i}")
    print(f"[{i+1}/{len(eval_rows)}] {trace_id}...", end=" ", flush=True)

    reference = next(
        p["text"]
        for m in row["messages"] if m["role"] == "assistant"
        for p in m["content"] if p["type"] == "text"
    )

    try:
        model_output = run_ollama(row, DATASET_ROOT)
        print("done")
    except Exception as e:
        model_output = f"(error: {e})"
        print(f"error: {e}")

    results.append({
        "trace_id": trace_id,
        "reference": reference,
        "model_output": model_output,
        "row": row,
    })

### 4.4 Compare outputs

In [ ]:
def show_comparison(result, dataset_root):
    """Display images, reference response, and model output side by side."""
    row = result["row"]
    display(Markdown(f"---\n### Trace `{result['trace_id']}`"))

    # Show images
    user_msg = next(m for m in row["messages"] if m["role"] == "user")
    images = [p for p in user_msg["content"] if p["type"] == "image"]
    if images:
        n = len(images)
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
        if n == 1:
            axes = [axes]
        for ax, img_part in zip(axes, images):
            img = Image.open(dataset_root / img_part["image"])
            ax.imshow(img)
            ax.set_title(Path(img_part["image"]).name)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    # Side-by-side text
    ref = result["reference"]
    out = result["model_output"]

    display(HTML(
        '<div style="display: flex; gap: 20px;">'
        '<div style="flex: 1; border: 1px solid #ccc; padding: 10px; border-radius: 8px;">'
        '<h4 style="color: #4A90D9;">Reference (Sonnet)</h4>'
        f'<pre style="white-space: pre-wrap; font-size: 12px;">{ref}</pre>'
        '</div>'
        '<div style="flex: 1; border: 1px solid #ccc; padding: 10px; border-radius: 8px;">'
        '<h4 style="color: #50C878;">Fine-tuned Gemma 4 E4B</h4>'
        f'<pre style="white-space: pre-wrap; font-size: 12px;">{out}</pre>'
        '</div>'
        '</div>'
    ))

In [ ]:
for r in results:
    show_comparison(r, DATASET_ROOT)

### 4.5 Section-level accuracy

Check whether the fine-tuned model outputs the expected structured sections:

In [ ]:
rows_data = []

for r in results:
    ref_sections = {s for s in SECTIONS if f"{s}:" in r["reference"]}
    out_sections = {s for s in SECTIONS if f"{s}:" in r["model_output"]}

    rows_data.append({
        "trace_id": r["trace_id"][:8],
        **{f"ref_{s}": s in ref_sections for s in SECTIONS},
        **{f"out_{s}": s in out_sections for s in SECTIONS},
        "sections_match": ref_sections == out_sections,
    })

accuracy_df = pd.DataFrame(rows_data)

match_rate = accuracy_df["sections_match"].mean() * 100
print(f"Section structure match rate: {match_rate:.0f}%\n")

for s in SECTIONS:
    ref_count = accuracy_df[f"ref_{s}"].sum()
    out_count = accuracy_df[f"out_{s}"].sum()
    print(f"  {s:16s}  ref={ref_count}  model={out_count}")

## 5. Optional: compare base vs fine-tuned

If you also have the stock Gemma 4 E4B in Ollama (`ollama pull gemma4:e4b`), run both to see the delta:

In [ ]:
BASE_MODEL = "gemma4:e4b"  # stock model name in Ollama
RUN_BASE_COMPARISON = False  # set True to run (slower — loads two models)

if RUN_BASE_COMPARISON:
    for i, row in enumerate(eval_rows[:2]):
        trace_id = row.get("trace_id", f"row-{i}")
        display(Markdown(f"---\n### Trace `{trace_id}`"))

        ref = next(
            p["text"]
            for m in row["messages"] if m["role"] == "assistant"
            for p in m["content"] if p["type"] == "text"
        )

        base_out = run_ollama(row, DATASET_ROOT, model_name=BASE_MODEL)
        tuned_out = run_ollama(row, DATASET_ROOT, model_name=OLLAMA_MODEL_NAME)

        display(HTML(
            '<div style="display: flex; gap: 12px;">'
            '<div style="flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 6px;">'
            '<h4 style="color: #999;">Reference (Sonnet)</h4>'
            f'<pre style="white-space: pre-wrap; font-size: 11px;">{ref}</pre></div>'
            '<div style="flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 6px;">'
            '<h4 style="color: #E8A040;">Base Gemma 4 E4B</h4>'
            f'<pre style="white-space: pre-wrap; font-size: 11px;">{base_out}</pre></div>'
            '<div style="flex: 1; border: 1px solid #ccc; padding: 8px; border-radius: 6px;">'
            '<h4 style="color: #50C878;">Fine-tuned</h4>'
            f'<pre style="white-space: pre-wrap; font-size: 11px;">{tuned_out}</pre></div>'
            '</div>'
        ))
else:
    print("Skipped. Set RUN_BASE_COMPARISON = True to run.")